# Building a Domain-Aligned Legal AI Assistant with Nugen

This cookbook demonstrates how to transform a general-purpose LLM into a
domain-aligned, benchmarked, production-ready model using Nugen APIs.


## End-to-End Flow

Document Upload → Benchmark Creation → Domain Alignment → Deployment → Inference


# Setup

In [ ]:
import requests

BASE_URL = "https://api.nugen.in"
API_KEY ="NUGEN_API_KEY"

HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}


# Upload Document

In [ ]:
upload_url = f"{BASE_URL}/api/v3/documents/"
files = {
    "files": (
        "file_name",
        open("FILE_NAME.txt", "rb"),
        "text/plain",
    )
}

upload_headers = {"Authorization": f"Bearer {API_KEY}"}

response = requests.post(upload_url, headers=upload_headers, files=files)
response.raise_for_status()

doc_id = response.json()["documents"][0]
print("Document ID:", doc_id)


# Create Benchmark

In [ ]:
benchmark_url = f"{BASE_URL}/api/v3/benchmark/create"

payload = {
    "documents": [doc_id],
    "num_questions": 10
}

response = requests.post(benchmark_url, headers=HEADERS, json=payload)
response.raise_for_status()

benchmark_id = response.json()["benchmark_id"]
print("Benchmark ID:", benchmark_id)


# Poll Benchmark Status

In [ ]:
status_url = f"{BASE_URL}/api/v3/benchmark/status/{benchmark_id}"

while True:
    status = requests.get(status_url, headers=HEADERS).json()
    print("Benchmark status:", status["status"])
    if status["status"] == "completed":
        break
    time.sleep(10)


# Create Alignment

In [ ]:
alignment_url = f"{BASE_URL}/api/v3/alignment-project/create"

payload = {
    "name": "legal-domain-alignment",
    "baseModel": "nugen-flash-instruct",
    "benchmarkId": benchmark_id,
    "document_ids": [doc_id],
    "description": "Legal domain aligned model"
}

response = requests.post(alignment_url, headers=HEADERS, json=payload)
response.raise_for_status()

alignment_id = response.json()["alignment_id"]
print("Alignment ID:", alignment_id)


# Poll Alignment

In [ ]:
status_url = f"{BASE_URL}/api/v3/alignment-project/status/{alignment_id}"

while True:
    status = requests.get(status_url, headers=HEADERS).json()
    print("Alignment status:", status["status"])
    if status["status"] == "completed":
        model_id = status["model_id"]
        break
    time.sleep(20)

print("Aligned Model ID:", model_id)


# Deploy Model

In [ ]:
deploy_url = f"{BASE_URL}/api/v3/models/deploy-model/{model_id}"

requests.post(deploy_url, headers=HEADERS)

status_url = f"{BASE_URL}/api/v3/models/deploy-model/{model_id}/status"

while True:
    status = requests.get(status_url, headers=HEADERS).json()
    print("Deployment:", status["status"])
    if status["status"] == "deployed":
        break
    time.sleep(10)


# Inference

In [ ]:
chat_url = f"{BASE_URL}/api/v3/inference/chat/completions"

payload = {
    "model": model_id,
    "messages": [
        {"role": "user", "content": "What is the purpose of the Equal Remuneration Act?"}
    ],
    "max_tokens": 300,
    "temperature": 0.2
}

response = requests.post(chat_url, headers=HEADERS, json=payload)
print(response.json())


# Troubleshooting

### Common Issues

• 401 Unauthorized → Check API key  
• 422 Validation Error → Payload mismatch  
• Model not responding → Ensure deployment status is `deployed`


# Key Takeaways

• Domain alignment reduces hallucinations  
• Benchmarks introduce measurable trust  
• Nugen APIs enable full automation  
• Suitable for regulated & enterprise AI